# ML-07 — Baseline Action Score and Top-10 Review

This baseline ranks pages for **content-refresh review** using only information available at the end of February 2026. March is the outcome window from W03 and is used only to audit the signals/rule, never as an input to the action score.

The rule is intentionally simple and auditable: **prioritise stale pages whose February search performance is weak relative to their exposure**. This is a baseline for Week 5, not a causal model.

## 1. My rule and its reason codes

### Two signal checks

I will test these two signals before encoding the rule:

1. **Content age / staleness** — this is the flag-linked signal. It is the simple age proxy behind refresh/staleness flags: older content should be more plausible refresh territory.
2. **February CTR** — low click efficiency should identify pages getting impressions without converting much of that exposure into clicks. This is a direct search-performance signal.

For each signal I will print a bucket table with `n` and the March `went_dark` rate, then give a one-word verdict: `CONFIRMED`, `OPPOSITE`, `MIXED`, or `FALSE`. The March outcome is audit-only and is never fed into the rule.

### Rule

For every page in the W03 decision universe:

- **+2 points** if `content_age_days >= 365` (stale enough to merit review).
- **+2 points** if `feb_ctr < 0.01` (low click efficiency).
- **+1 point** if `feb_position >= 10` (weak average search position).

The queue is sorted by this score descending, then February clicks ascending as a tie-breaker. The action is:

- score >= 4 → `REFRESH_REVIEW`
- score 2–3 → `WATCH`
- score < 2 → `NO_ACTION`

Each row gets **one** reason code: the highest-priority condition that contributed to its score (`STALE_LOW_CTR`, `STALE`, `LOW_CTR`, `WEAK_POSITION`, or `NO_SIGNAL`). The reason code is explanatory, not a second score.

In [ ]:
import os
import duckdb
import pandas as pd
import numpy as np

try:
    from google.colab import userdata
except ImportError:
    userdata = None

hf_token = os.environ.get("HF_TOKEN")
if not hf_token and userdata is not None:
    hf_token = userdata.get("HF_TOKEN")
if not hf_token:
    raise RuntimeError("Set HF_TOKEN as a Colab Secret or environment variable before running this notebook.")

con = duckdb.connect()
con.execute("SET enable_progress_bar = false")
con.execute("SET VARIABLE hf_token = ?", [hf_token])
con.execute("CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN getvariable('hf_token'))")

REL = "hf://datasets/FlyRank/internship-warehouse"
FACT = f"{REL}/fact_content_daily_performance"
DIM = f"{REL}/dim_content.parquet"
FEB = f"read_parquet('{FACT}/month=2026-02/*.parquet')"
MAR = f"read_parquet('{FACT}/month=2026-03/*.parquet')"

print("Ready: February = decision window; March = audit-only outcome window.")

In [ ]:
# Build the W03 decision universe and February-only signals.
# March is joined separately below only for signal validation.
base = con.sql(f"""
WITH feb AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) FILTER (WHERE gsc_data_available IS TRUE) AS feb_impressions,
        SUM(gsc_clicks) FILTER (WHERE gsc_data_available IS TRUE) AS feb_clicks,
        SUM(gsc_sum_position) FILTER (WHERE gsc_data_available IS TRUE)
          / NULLIF(SUM(gsc_impressions) FILTER (WHERE gsc_data_available IS TRUE), 0) AS feb_position
    FROM {FEB}
    GROUP BY client_hash_id, content_hash_id
    HAVING SUM(gsc_impressions) FILTER (WHERE gsc_data_available IS TRUE) >= 100
       AND SUM(gsc_clicks) FILTER (WHERE gsc_data_available IS TRUE) >= 3
),
march AS (
    SELECT client_hash_id, content_hash_id,
           SUM(gsc_clicks) FILTER (WHERE gsc_data_available IS TRUE) AS march_clicks,
           COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS march_measured_days
    FROM {MAR}
    GROUP BY client_hash_id, content_hash_id
)
SELECT
    f.client_hash_id,
    f.content_hash_id,
    f.feb_impressions,
    f.feb_clicks,
    f.feb_clicks / NULLIF(f.feb_impressions, 0) AS feb_ctr,
    f.feb_position,
    DATE_DIFF('day', d.content_created_date, DATE '2026-02-28') AS content_age_days,
    COALESCE(m.march_clicks, 0) AS march_clicks,
    COALESCE(m.march_measured_days, 0) AS march_measured_days
FROM feb f
JOIN {DIM} d USING (client_hash_id, content_hash_id)
LEFT JOIN march m USING (client_hash_id, content_hash_id)
WHERE d.is_published IS TRUE
  AND d.content_created_date <= DATE '2026-02-28'
""").df()

# A March outcome is used only for the two signal audits, not the baseline inputs.
auditable = base[base["march_measured_days"] > 0].copy()
auditable["went_dark"] = (auditable["march_clicks"] == 0).astype(int)

print(f"Decision-universe rows: {len(base):,}")
print(f"Rows with a measured March outcome for signal audit: {len(auditable):,}")

In [ ]:
# SIGNAL 1: content age / staleness (flag-linked)
# Bucket boundaries are simple and pre-declared: <180, 180–364, >=365 days.
age_audit = auditable.copy()
age_audit["age_bucket"] = pd.cut(
    age_audit["content_age_days"],
    bins=[-np.inf, 179, 364, np.inf],
    labels=["<180d", "180-364d", "365d+"]
)
age_table = (
    age_audit.groupby("age_bucket", observed=False)
    .agg(n=("went_dark", "size"), went_dark_rate=("went_dark", "mean"))
    .reset_index()
)
age_table["went_dark_rate"] = age_table["went_dark_rate"].round(3)
print(age_table.to_string(index=False))

# Verdict: CONFIRMED only if the observed rate rises monotonically with age.
rates = age_table["went_dark_rate"].tolist()
age_verdict = "CONFIRMED" if len(rates) == 3 and rates[0] <= rates[1] <= rates[2] else "MIXED"
print("Verdict:", age_verdict)

In [ ]:
# SIGNAL 2: February CTR
# Bucket boundaries: <1%, 1–2%, >=2%.
ctr_audit = auditable.copy()
ctr_audit["ctr_bucket"] = pd.cut(
    ctr_audit["feb_ctr"],
    bins=[-np.inf, 0.01, 0.02, np.inf],
    labels=["<1%", "1-2%", ">=2%"]
)
ctr_table = (
    ctr_audit.groupby("ctr_bucket", observed=False)
    .agg(n=("went_dark", "size"), went_dark_rate=("went_dark", "mean"))
    .reset_index()
)
ctr_table["went_dark_rate"] = ctr_table["went_dark_rate"].round(3)
print(ctr_table.to_string(index=False))

rates = ctr_table["went_dark_rate"].tolist()
ctr_verdict = "CONFIRMED" if len(rates) == 3 and rates[0] >= rates[1] >= rates[2] else "MIXED"
print("Verdict:", ctr_verdict)

### Why these checks matter

The rule is allowed to be simple only after checking that its signals have an observed directional relationship with the outcome. A `MIXED` verdict is not a failure: it is evidence to weaken, remove, or qualify that signal rather than pretending the pattern is stronger than it is.

The March `went_dark` rate above is **audit data only**. It is not included in the CSV score and is not available at the February decision point.

## 2. Build the ranked queue (writes the CSV)

The queue uses **February-only inputs**. No March clicks, March impressions, `went_dark`, optimization fields, or other future-window information enters the score.

In [ ]:
# Encode ONE deterministic baseline rule.
queue = base.copy()

queue["score"] = (
    (queue["content_age_days"] >= 365).astype(int) * 2
    + (queue["feb_ctr"] < 0.01).astype(int) * 2
    + (queue["feb_position"] >= 10).astype(int)
)

queue["action"] = np.select(
    [queue["score"] >= 4, queue["score"] >= 2],
    ["REFRESH_REVIEW", "WATCH"],
    default="NO_ACTION"
)

queue["reason_code"] = np.select(
    [
        (queue["content_age_days"] >= 365) & (queue["feb_ctr"] < 0.01),
        queue["content_age_days"] >= 365,
        queue["feb_ctr"] < 0.01,
        queue["feb_position"] >= 10,
    ],
    ["STALE_LOW_CTR", "STALE", "LOW_CTR", "WEAK_POSITION"],
    default="NO_SIGNAL"
)

queue = queue.sort_values(
    ["score", "feb_clicks", "feb_impressions", "content_hash_id"],
    ascending=[False, True, False, True]
).reset_index(drop=True)
queue.insert(0, "rank", np.arange(1, len(queue) + 1))

output_cols = [
    "rank", "client_hash_id", "content_hash_id", "score", "reason_code", "action",
    "content_age_days", "feb_impressions", "feb_clicks", "feb_ctr", "feb_position"
]
output = queue[output_cols].copy()

os.makedirs("work/outputs", exist_ok=True)
out_path = "work/outputs/baseline_action_score.csv"
output.to_csv(out_path, index=False)

print(f"Wrote {len(output):,} rows to {out_path}")
print("Action counts:")
print(output["action"].value_counts().to_string())
print("\nScore distribution:")
print(output["score"].value_counts().sort_index().to_string())
output.head(10)

## 3. Top-10 review

For every top-ten row, the review records the action, why the rule selected it, and a concrete condition that would make the recommendation wrong. The last column is deliberately skeptical: a high baseline score is not proof that a refresh is needed.

In [ ]:
top10 = output.head(10).copy()

def why(row):
    parts = []
    if row["content_age_days"] >= 365:
        parts.append("content is at least 365 days old")
    if row["feb_ctr"] < 0.01:
        parts.append("February CTR is below 1%")
    if row["feb_position"] >= 10:
        parts.append("February position is 10 or worse")
    return "; ".join(parts) if parts else "no strong rule signal"

def wrong_if(row):
    if row["content_age_days"] >= 365 and row["feb_ctr"] < 0.01:
        return "The low CTR may be normal for the query mix or SERP layout, or the page may already be intentionally stable."
    if row["content_age_days"] >= 365:
        return "The page may be evergreen and still appropriate despite its age."
    if row["feb_ctr"] < 0.01:
        return "The low CTR may be explained by query intent or SERP features rather than stale content."
    return "Position-based weakness may be caused by competition or search demand rather than content quality."

top10["why"] = top10.apply(why, axis=1)
top10["what_would_make_it_wrong"] = top10.apply(wrong_if, axis=1)

review_cols = ["rank", "action", "reason_code", "score", "why", "what_would_make_it_wrong"]
print(top10[review_cols].to_string(index=False))

## 4. Weak picks + leakage check

The weakest picks are where the rule can be technically consistent but operationally misleading: a page can be old without being stale, or have low CTR because of query intent rather than content quality. Those are reasons to review the queue, not reasons to hide the rule.

The baseline contains no March feature, no `went_dark`, and no label-derived column. March appears only in Section 1's signal-audit tables. The CSV contains February decision-time signals only.

In [ ]:
# Show a few low-score rows that are likely weak operational picks.
weak = output[output["score"] <= 1].head(5).copy()
weak["weak_pick_reason"] = np.select(
    [weak["content_age_days"] >= 365, weak["feb_ctr"] < 0.01, weak["feb_position"] >= 10],
    [
        "Age alone is weak evidence that a refresh will help.",
        "Low CTR alone may reflect query intent or SERP layout.",
        "Position alone does not identify a content problem."
    ],
    default="Little direct evidence for action."
)
print(weak[["rank", "score", "reason_code", "action", "weak_pick_reason"]].to_string(index=False))

# Mechanical leakage guard: forbidden future/label fields must not be output features.
forbidden = {"march_clicks", "march_measured_days", "went_dark", "last_optimized_date", "optimization_eligible_date", "content_updated_date"}
leaked = forbidden.intersection(set(output.columns))
print("\nForbidden columns in final queue:", leaked)
assert not leaked, f"Leakage guard failed: {leaked}"
print("Leakage guard: PASS")

## 5. Self-check

- [x] Two signal checks have visible bucket tables with `n` and one-word verdicts.
- [x] At least one checked signal is explicitly flag-linked: content staleness/age.
- [x] One deterministic score, one reason code, and one action label are encoded.
- [x] The ranked queue is written to `work/outputs/baseline_action_score.csv`.
- [x] Top 10 are reviewed with action, reason, and “what would make it wrong”.
- [x] March outcome data is audit-only; no future-window or label-derived input enters the queue.
- [ ] Run the notebook end-to-end in Colab with `HF_TOKEN`, inspect outputs, and commit the executed notebook. 
- [ ] Keep the generated CSV out of git; the notebook regenerates it.